# Printed replay: expansion V1

Wybierz **CPU**, potem **Uruchom wszystko**. Niczego nie wgrywaj. Zamrozona partia: 34 stron, 8 rodzin zrodel. Osobne rodziny kontrolne nie trafiaja do treningu.

To wydobywanie kandydatow replay, **nie trening**. Zachowujemy stara pisownie. Dane obejmuja dawny druk, nie reprezentatywny zbior wspolczesnych dokumentow.

## 1. Srodowisko i testy

In [ ]:
from pathlib import Path
import hashlib
import os
import subprocess
import sys
import venv
from datetime import datetime, timezone

CODE_REVISION = 'c8ae68ae741d2c30d95317c2c0bb84d347d79a16'
DATA_REVISION = '5fc2e96fdfc2df83a386c80ffb765bb22e1030e8'
DATA_SHA256 = '95661ff30f5b904f67896e8079d8332d4665cdf9b9b3d579bc4a2e3f6d548769'
DATA_PATH = 'data/printed-replay-source-expansion-v1-20261007/printed-replay-source-expansion-v1.zip'
repo = Path('/content/OCR_engine-printed-replay')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/PiotrStyla/OCR_engine.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', CODE_REVISION], check=True)
assert subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip() == CODE_REVISION
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'djvulibre-bin', 'tesseract-ocr', 'tesseract-ocr-pol'], check=True)
environment = Path('/content/slayer-printed-replay-env')
python = environment/'bin/python'
if not python.exists():
    venv.EnvBuilder(with_pip=False, system_site_packages=True).create(environment)
subprocess.run([str(python), '-m', 'pip', 'install', '--no-cache-dir',
                'pillow==11.3.0', 'huggingface_hub==0.36.2', 'beautifulsoup4==4.13.5',
                'pytest==8.4.2', 'nbformat==5.10.4'], check=True)
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['PYTEST_DISABLE_PLUGIN_AUTOLOAD'] = '1'
subprocess.run([str(python), '-m', 'pytest', 'tests/test_printed_replay_pilot.py', 'tests/test_printed_replay_expansion.py',
                '-q', '-p', 'no:cacheprovider'], cwd=repo, check=True)
work = Path('/content/printed-replay-expansion-v1-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
work.mkdir()


## 2. Automatyczne pobranie sprawdzonych zrodel

In [ ]:
import zipfile
from pathlib import PurePosixPath
download_code = """from huggingface_hub import hf_hub_download
import sys
print(hf_hub_download('PiotrSty/slayer-ocr-datasets', sys.argv[2], repo_type='dataset', revision=sys.argv[1]))
"""
archive = Path(subprocess.check_output([str(python), '-c', download_code, DATA_REVISION, DATA_PATH], text=True).strip())
assert hashlib.sha256(archive.read_bytes()).hexdigest() == DATA_SHA256, 'Checksum danych nie zgadza sie.'
source_root = work/'input'
source_root.mkdir()
with zipfile.ZipFile(archive) as zipped:
    members = zipped.infolist()
    assert len({m.filename for m in members}) == len(members), 'Duplicate ZIP member'
    assert sum(m.file_size for m in members) <= 256_000_000, 'Oversized input archive'
    for member in members:
        path = PurePosixPath(member.orig_filename)
        assert member.orig_filename == member.filename and ':' not in member.filename
        assert not path.is_absolute() and '..' not in path.parts and '\\' not in member.filename
        assert ((member.external_attr >> 16) & 0o170000) != 0o120000, 'Symlink w ZIP'
    zipped.extractall(source_root)
print('INPUT_READY', source_root)

WORKS_CONFIG = repo/'experiments/2026-10-07/printed-replay-expansion-v1/works.json'
assert hashlib.sha256(WORKS_CONFIG.read_bytes()).hexdigest() == 'ee01b99977b100d3eb3a309c0e59fb9f3d762017d74b66ccdac048a4789f4b93', 'Work config checksum mismatch'


## 3. Native skany, linie i ZIP

Zwracany plik: `printed-replay-expansion-v1-evidence.zip`. Zawiera wszystkie native strony, TSV, wycinki, odrzucenia i pochodzenie. Jesli automatyczne pobieranie jest blokowane, pobierz ZIP z panelu Pliki w katalogu printed-replay-expansion-v1-...; nie zamykaj sesji wczesniej.

In [ ]:
from google.colab import files
EVIDENCE_ZIP = None
command = [str(python), '-m', 'training.mine_printed_replay', '--source-root', str(source_root),
           '--output', str(work/'mining'), '--protocol', 'v2', '--works-config', str(WORKS_CONFIG)]
process = subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
    if line.startswith('EVIDENCE_ZIP '):
        EVIDENCE_ZIP = Path(line.strip().split(' ', 1)[1])
returncode = process.wait()
assert returncode == 0, 'Ekstrakcja nie zakonczona. Wklej blad; nie uruchamiaj treningu.'
assert EVIDENCE_ZIP is not None and EVIDENCE_ZIP.is_file(), 'Brak kompletnego ZIP-a.'
import shutil
export_zip = work/'printed-replay-expansion-v1-evidence.zip'
shutil.copyfile(EVIDENCE_ZIP, export_zip)
EVIDENCE_ZIP = export_zip
print('Gotowy ZIP:', EVIDENCE_ZIP)
files.download(str(EVIDENCE_ZIP))
